## Import packages

In [ ]:
import os, sys
import pandas as pd
import numpy as np
import matplotlib
from pym3c.hic import mcds2frac
from scipy.stats import zscore, pearsonr, norm
from concurrent.futures import ProcessPoolExecutor, as_completed
import cooler
import warnings
warnings.filterwarnings("ignore")
import matplotlib.pylab as plt
import matplotlib as mpl
import seaborn as sns
import pybedtools
pybedtools.helpers.set_bedtools_path(path=os.path.dirname(sys.executable))
mpl.style.use('default')
mpl.rcParams['pdf.fonttype'] = 42
mpl.rcParams['ps.fonttype'] = 42
mpl.rcParams['font.family'] = 'sans-serif'
mpl.rcParams['font.sans-serif'] = 'Arial'
mpl.rcParams['figure.dpi'] = 80
mpl.rcParams['savefig.dpi']=300
from PyComplexHeatmap import *

## Parameters

In [ ]:
workdir = "/home/x-wding2/project_wubin/BG/HIC/HBA_pseudobulk"
indir = None
res=10000
chrom_size_path = "/home/x-wding2/Ref/hg38/hg38_ucsc.nochrM.sizes"
metadata_path = "/home/x-wding2/Projects/BG/clustering/100kb/cell_metadata_with_annotation.tsv"
pseudobulk_mcds_path = "/home/x-wding2/project_wubin/BG/pseudobulk/Leaf/Study+Name+donor+Region+Subclass.5kb.mcds" #"5kb", used to calculate correlation between mC and compartment score
pseudobulk_atac_adata_path=None
var_dim5kb="5kb"
extract_group=None
query_group=None
normalize_per_cell=True
loop_prefix='Subclass' # Subclass+Region
group_col='Subclass' # Region # used to define figdir, and cell_types (group_order), match the col in metadata.columns
parent_col="Class" #used to sort the group_col (MajorType)
sort_cols=None
stratify_col = None #Subclass
palette_path = "/home/x-wding2/Projects/BG/metadata/HMBA_color_palette.xlsx"
split_order = ["Nonneuron", "Glut", "Splatter", "Subpallium GABA"]
figdir=None
cpu = 16
gene_annotation=None
marker_path=None
marker_sheet=None
species='human'
kernel_name = "m3c"
excluded_groups=['Low quality cells','doublet','Unk']

extract_group = [4]
query_group = {"0": "HBA"}

extract_group = [4,3]
group_col = "Region"
loop_prefix="Subclass+Region"
stratify_col = "Subclass"

In [ ]:
# Parameters
workdir = "/home/x-wding2/Projects/BG/HIC/BG_Class"
metadata_path = "/home/x-wding2/Projects/BG/clustering/100kb/annotations_without_Glut.tsv"
adata_prefix = "/home/x-wding2/Projects/BG/adata/BG"
atac_adata_path = "/home/x-wding2/Projects/BG/atac/HMBA.atac-5kb.h5ad"
atac_ann_path = "/home/x-wding2/Projects/BG/atac/obs.tsv"
pseudobulk_mcds_path = "/home/x-wding2/Projects/BG/pseudobulk/Class/Class.mcds"
pseudobulk_atac_adata_path = "/home/x-wding2/Projects/BG/atac/Pseudobulk.Class.5kb.h5ad"
chrom_size_path = "/home/x-wding2/Ref/hg38/hg38_ucsc.nochrM.sizes"
palette_path = "/home/x-wding2/Projects/BG/metadata/BG_color_palette.xlsx"
cg_count_path = "/home/x-wding2/Ref/hg38/annotations/CG-count-100k.tsv"
gene_annotation = "/home/x-wding2/Ref/hg38/annotations/gene_2000_2000.bed.gz"
marker_path = "/home/x-wding2/Projects/BG/metadata/BG_markers.xlsx"
marker_sheet = "BG"
mode = "raw"
prefix = "Class"
group_col = "Class"
parent_col = "Neighborhood"
sort_cols = ["Neighborhood"]
stratify_col = None
loop_prefix = None
query_str = None
excluded_groups = ["Low quality cells", "doublet", "Unk", "Glut"]
figdir = None
cpu = 8
indir = None
kernel_name = "m3c"
genome = "hg38"
obs_dim = "cell"
var_dim = "100kb"
var_dim5kb = "5kb"
extract_group = None
query_group = None
mc_types = ["CGN", "CHN"]
normalize_per_cell = True
split_order = ["Nonneuron", "Subpallium GABA", "Glut Sero Dopa", "Subpallium GABA-Glut"]
diff_comp_dir = None
dchic_env = "dchic"
param_yaml = "/home/x-wding2/Projects/Github/pym3c/pym3c/notebooks/templates/BG_Subclass_hic.yaml"
outdir = "/home/x-wding2/Projects/BG/HIC/BG_Class/loop"
outname = "2.Class.loop_correlation.ipynb"


In [ ]:
if loop_prefix is None:
    loop_prefix=group_col
if not workdir is None:
    workdir=os.path.expanduser(workdir)
if not indir is None:
    indir=os.path.expanduser(indir)
else:
    assert not workdir is None
    indir=os.path.join(workdir,f"{loop_prefix}_loop")
if figdir is None:
    assert not workdir is None
    figdir=os.path.join(workdir,"loop",group_col)
else:
    figdir=os.path.expanduser(figdir)
if not os.path.exists(figdir):
    os.makedirs(figdir, exist_ok=True)
chrom_size_path=os.path.expanduser(chrom_size_path)
metadata_path=os.path.expanduser(metadata_path)
chrom_size_path=os.path.expanduser(chrom_size_path)
print(indir,figdir)

In [ ]:
chrom_sizes = cooler.read_chromsizes(chrom_size_path, all_names=True)
keep_chroms=[chrom for chrom in chrom_sizes.index if chrom not in ['chrX','chrY']]
chrom_sizes = chrom_sizes[keep_chroms]
df_chr=chrom_sizes.to_frame()
df_chr['N']=df_chr.index.to_series().apply(lambda x:int(x.lstrip('chr')) if x.lstrip('chr') not in ['X','Y'] else 30) #sort by chrom int (1,2,3..)
df_chr.sort_values('N',inplace=True)
chrom_sizes=chrom_sizes.loc[df_chr.index.tolist()]
if metadata_path.endswith('.csv') or metadata_path.endswith('.csv.gz'):
    metadata = pd.read_csv(metadata_path, index_col=0)
else:
    metadata = pd.read_csv(metadata_path, index_col=0,sep='\t')
# Filter out low quality cells
metadata=metadata.loc[~ (metadata[group_col].isin(excluded_groups))]
metadata=metadata.loc[metadata[group_col].apply(lambda x:not str(x).endswith('-Unk'))]
if not stratify_col is None:
    metadata=metadata.loc[~ (metadata[stratify_col].isin(excluded_groups))]
if not parent_col is None:
    metadata=metadata.loc[~ (metadata[parent_col].isin(excluded_groups))]
if sort_cols is None:
    sort_cols=parent_col
metadata.head()

In [ ]:
# define group_order (such as cell types) and stratify2group (such as Subclass to Subclass+Region)
if not sort_cols is None:
    metadata.sort_values(sort_cols,inplace=True)
else:
    metadata.sort_values(group_col,inplace=True)
sorted_groups=metadata[group_col].unique().tolist()
stratify2group={}
group_order=[]
if stratify_col is None:
    for g in sorted_groups:
        sname=g.replace(' ','_')
        infile=f'{indir}/{sname}/{sname}/{sname}.loop.bedpe'
        if os.path.exists(infile):
            with open(infile,'r') as f:
                N=len(f.readlines())
            if N>0:
                group_order.append(g)
else:
    metadata['GroupID']=metadata[stratify_col].map(str)+'+'+metadata[group_col].map(str)
    stratify_values=metadata[stratify_col].unique().tolist()
    for s in stratify_values:
        s_name=s.replace(' ','_')
        for g in sorted_groups:
            g_name=g.replace(' ','_')
            ct=f'{s_name}+{g_name}'
            infile=os.path.join(indir,s_name,ct,f'{ct}.loop.bedpe')
            if os.path.exists(infile):
                if s not in stratify2group:
                    stratify2group[s]=[]
                with open(infile,'r') as f:
                    N=len(f.readlines())
                if N>0:
                    stratify2group[s].append(g) # For example: keys are Subclass, values are Subclass+Region
                    group_order.append(f"{s}+{g}") # such as Subclass+Region: 'Microglia+CaB', 'Microglia+NAC','OPC+CaB'

if not parent_col is None:
    parent2group=metadata.loc[:,[parent_col,group_col]].drop_duplicates().groupby(parent_col)[group_col].apply(lambda x:x.unique().tolist()).to_dict()
else:
    parent2group={}
parent2group['merged']=group_order
print(len(group_order),group_order)
print(stratify2group)
print(parent2group)

In [ ]:
# read color palette
D=pd.read_excel(os.path.expanduser(palette_path),
                sheet_name=None, index_col=0)
D1=D[group_col].Hex.to_dict()
group_colors={k: D1[k] for k in D1 if k in sorted_groups}
if not parent_col is None:
    parent_colors=D[parent_col].Hex.to_dict()
    if stratify_col is None:
        group2parent=metadata.loc[:,[group_col,parent_col]].drop_duplicates().set_index(group_col)[parent_col].to_dict()
    else:
        D1=D[stratify_col].Hex.to_dict()
        stratify_colors={k: D1[k] for k in D1 if k in stratify_values}
        group2parent=metadata.loc[:,[parent_col,'GroupID']].drop_duplicates().set_index('GroupID')[parent_col].to_dict()
    group2parent['merged']='merged'
else:
    parent_colors=None
    group2parent=None
print(parent_colors)
print(group_colors)
print(group2parent)

## Load data

In [ ]:
df_loop=pd.read_csv(os.path.join(figdir,'merged_loop_with_anova.tsv'),sep='\t')
df_loop

In [ ]:
idx1 = df_loop['chr1'] + '_' + (df_loop['start1'] // res).astype(str) #bin ID for start1 position
idx2 = df_loop['chr2'] + '_' + (df_loop['start2'] // res).astype(str) # bin ID for start2 position
idx1[:5],idx2[:5]

In [ ]:
loopq = pd.read_csv(os.path.join(figdir,f'merged_loop_Q.tsv'),sep='\t')
# loopt = pd.read_csv(os.path.join(figdir,f'merged_loop_T.tsv'),sep='\t')
loopq=loopq.loc[:,group_order] #make sure loop have the same cell types order as group_order
# loopt=loopt.loc[:,group_order]
# df_loop, loopq and loopt have the same rows and row orders.
print(loopq.shape)
loopq.head()

In [ ]:
frac_dict=mcds2frac(pseudobulk_mcds_path=pseudobulk_mcds_path,
					   var_dim=var_dim5kb,res=res,mc_types=['CGN','CHN'],
					   extract_group=extract_group,query_group=query_group,filter_chroms=True,
					   normalize_per_cell=normalize_per_cell)

In [ ]:
# mCG
mcg=frac_dict['CGN']
use_cols=list(set(mcg.columns.tolist()) & set(group_order))
mcg = mcg.loc[:,use_cols]
# mcg.head() #methylation fraction for each 100kb bin in each cell type

In [ ]:
loop_avg_cg = (mcg.reindex(index=idx1.values).values + mcg.reindex(index=idx2.values).values) / 2 #average methylation between bin1 and bin2 for each loop
loop_avg_cg = pd.DataFrame(loop_avg_cg, index=loopq.index, columns=mcg.columns.tolist())
loop_avg_cg

In [ ]:
def calculate_pearsonr(df1,df2):
    return [pearsonr(xx,yy)[0] for xx,yy in zip(df1.values, df2.values)]

In [ ]:
assert loopq.index.tolist() == loop_avg_cg.index.tolist()
usecols=list(set(loopq.columns.tolist()) & set(loop_avg_cg.columns.tolist()))
# df_loop['mCG_corr'] = [pearsonr(xx,yy)[0] for xx,yy in zip(loop_avg_cg.values, loopq.values)] #correlation between loop avg mC and Q
## it's too slow using only one CPU
with ProcessPoolExecutor(cpu) as executor:
    futures = {}
    indexes=[]
    chunksize=100000
    i=0
    while i < loopq.shape[0]:
        future = executor.submit(calculate_pearsonr,
            df1=loop_avg_cg.loc[:,usecols].iloc[i:i+chunksize],df2=loopq.loc[:,usecols].iloc[i:i+chunksize]
        )
        futures[future] = i
        indexes.append(i)
        i+=chunksize
    print(indexes,len(indexes))
    result = {}
    for future in as_completed(futures):
        i = futures[future]
        result[i] = future.result()
        print(i,end=',')
coef=[]
for i in indexes:
    coef.extend(result[i])
df_loop['mCG_corr']=coef
df_loop.head()

In [ ]:
# mCH
mch=frac_dict['CHN']
use_cols=list(set(mch.columns.tolist()) & set(group_order))
mch = mch.loc[:,use_cols]

In [ ]:
loop_avg_ch = (mch.reindex(index=idx1.values).values + mch.reindex(index=idx2.values).values) / 2 #average methylation between bin1 and bin2 for each loop
loop_avg_ch = pd.DataFrame(loop_avg_ch, index=loopq.index, columns=mch.columns.tolist())
loop_avg_ch

In [ ]:
usecols=list(set(loopq.columns.tolist()) & set(loop_avg_cg.columns.tolist()))
assert loopq.index.tolist() == loop_avg_ch.index.tolist()
# df_loop['mCH_corr'] = [pearsonr(xx,yy)[0] for xx,yy in zip(loop_avg_ch.values, loopq.values)] #correlation between loop avg mC and Q
## it's too slow using only one CPU
with ProcessPoolExecutor(cpu) as executor:
    futures = {}
    indexes=[]
    chunksize=100000
    i=0
    while i < loopq.shape[0]:
        future = executor.submit(calculate_pearsonr,
            df1=loop_avg_ch.loc[:,usecols].iloc[i:i+chunksize],df2=loopq.loc[:,usecols].iloc[i:i+chunksize]
        )
        futures[future] = i
        indexes.append(i)
        i+=chunksize
    print(indexes,len(indexes))
    result = {}
    for future in as_completed(futures):
        i = futures[future]
        result[i] = future.result()
        print(i,end=',')
coef=[]
for i in indexes:
    coef.extend(result[i])
df_loop['mCH_corr']=coef
df_loop.head()

In [ ]:
from pym3c.adata import merge_adata_regions
if not pseudobulk_atac_adata_path is None:
    atac_data=merge_adata_regions(pseudobulk_atac_adata_path,bin_size=5000,use_raw=True,
    		res=res,filter_chroms=True)
    atac_data=atac_data.T
    print(atac_data.shape)
else:
    atac_data=None

In [ ]:
if not pseudobulk_atac_adata_path is None:
    use_cols=list(set(atac_data.columns.tolist()) & set(group_order))
    atac = atac_data.loc[:,use_cols]
    loop_avg_atac = (atac.reindex(index=idx1.values).values + atac.reindex(index=idx2.values).values) / 2 #average ATAC between bin1 and bin2 for each loop
    loop_avg_atac = pd.DataFrame(loop_avg_atac, index=loopq.index, columns=atac.columns.tolist())

In [ ]:
usecols=list(set(loopq.columns.tolist()) & set(loop_avg_atac.columns.tolist()))
assert loopq.index.tolist() == loop_avg_atac.index.tolist()
with ProcessPoolExecutor(cpu) as executor:
    futures = {}
    indexes=[]
    chunksize=100000
    i=0
    while i < loopq.shape[0]:
        future = executor.submit(calculate_pearsonr,
            df1=loop_avg_atac.loc[:,usecols].iloc[i:i+chunksize],df2=loopq.loc[:,usecols].iloc[i:i+chunksize]
        )
        futures[future] = i
        indexes.append(i)
        i+=chunksize
    print(indexes,len(indexes))
    result = {}
    for future in as_completed(futures):
        i = futures[future]
        result[i] = future.result()
        print(i,end=',')
coef=[]
for i in indexes:
    coef.extend(result[i])
df_loop['ATAC_corr']=coef
df_loop.head()

In [ ]:
df_loop.to_csv(os.path.join(figdir,'merged_loop_with_mc_correlation.tsv'),sep='\t',index=False)
loop_avg_cg.to_csv(os.path.join(figdir,'loop_avg_cg.tsv'),sep='\t',index=False)
loop_avg_ch.to_csv(os.path.join(figdir,'loop_avg_ch.tsv'),sep='\t',index=False)
if not atac_data is None:
    loop_avg_atac.to_csv(os.path.join(figdir,'loop_avg_atac.tsv'),sep='\t',index=False)

## Plot correlations

In [ ]:
df_loop=pd.read_csv(os.path.join(figdir,'merged_loop_with_mc_correlation.tsv'),sep='\t')
loop_avg_cg=pd.read_csv(os.path.join(figdir,'loop_avg_cg.tsv'),sep='\t')
loop_avg_ch=pd.read_csv(os.path.join(figdir,'loop_avg_ch.tsv'),sep='\t')
usecols=list(set(loop_avg_cg.columns.tolist()) & set(group_order))
loop_avg_cg=loop_avg_cg.loc[:,usecols]
loop_avg_ch=loop_avg_ch.loc[:,usecols]
if not pseudobulk_atac_adata_path is None:
    loop_avg_atac=pd.read_csv(os.path.join(figdir,'loop_avg_atac.tsv'),sep='\t')
    loop_avg_atac=loop_avg_atac.loc[:,usecols]

In [ ]:
loopq = pd.read_csv(os.path.join(figdir,f'merged_loop_Q.tsv'),sep='\t')
loopt = pd.read_csv(os.path.join(figdir,f'merged_loop_T.tsv'),sep='\t')
usecols=list(set(loopq.columns.tolist()) & set(group_order))
loopq=loopq.loc[:,usecols] #make sure loop have the same cell types order as group_order
loopt=loopt.loc[:,usecols]
# df_loop, loopq and loopt have the same rows and row orders.
print(loopq.shape)
loopq.head()

In [ ]:
df_loop.head()

In [ ]:
thres1 = norm.isf(0.025)
thres2 = norm.isf(0.15)
print(thres1, thres2)

In [ ]:
use_group=group_col if stratify_col is None else stratify_col
sig_loops1 = ((zscore(df_loop[f'{use_group}.Q.F'])>thres2) & (zscore(df_loop[f'{use_group}.T.F'])>thres2)) #for visualization, use thres2, otherwise, using thres1
print(sig_loops1.sum()) # significant diff loops
sig_loops2 = ((zscore(df_loop[f'{use_group}.Q.F']) < 0) & (zscore(df_loop[f'{use_group}.T.F']) < 0))
print(sig_loops2.sum()) # background?

In [ ]:
df_loop[f'{use_group}.Q.avg_group'] = pd.cut(zscore(loopq.mean(axis=1)), 100, labels=False)
df_loop[f'{use_group}.T.avg_group'] = pd.cut(zscore(loopt.mean(axis=1)), 100, labels=False)
df_loop['group'] = np.zeros(df_loop.shape[0])
df_loop.loc[sig_loops1, 'group'] = 1 # sig group
df_loop.loc[sig_loops2, 'group'] = -1 #background group

In [ ]:
df_loop.head()

In [ ]:
selected = {-1:[],1:[]}
for _,tmp in df_loop.loc[df_loop['group']!=0].groupby([f'{use_group}.Q.avg_group',f'{use_group}.T.avg_group']):
    t = tmp['group'].value_counts()
    if (len(t)==2) and t.min()>0:
        t = t.min()
        for group,xx in tmp.groupby('group'): #1 or -1, sig or background
            # print(i,group)
            np.random.seed(0)
            if xx.shape[0]>t:
                selected[group].append(np.random.choice(xx.index.values, t, False))
            else:
                selected[group].append(xx.index.values)
len(selected)

In [ ]:
selpos = np.concatenate(selected[1])
selneg = np.concatenate(selected[-1])
print(selpos.shape[0], selneg.shape[0])

In [ ]:
print(df_loop.loc[selpos,f'{use_group}.Q.F'].describe())
print(df_loop.loc[selneg,f'{use_group}.Q.F'].describe())

In [ ]:
df_loop['LoopType']=''
df_loop.loc[selpos,'BalancedGroup']='DiffLoop'
df_loop.loc[selneg,'BalancedGroup']='ConsLoop' #show no difference between cell types

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(5,2.5), dpi=300)

ax = axes[0]
sns.histplot(loopq.loc[sig_loops1].mean(axis=1), ax=ax,color='chocolate',alpha=0.6)
sns.histplot(loopq.loc[sig_loops2].mean(axis=1), ax=ax,color='cadetblue',alpha=0.6)
ax.set_xlabel('Q')
ax = axes[1]
sns.histplot(loopq.loc[selpos].mean(axis=1), ax=ax,color='chocolate',alpha=0.6)
sns.histplot(loopq.loc[selneg].mean(axis=1), ax=ax,color='cadetblue',alpha=0.6)
ax.set_xlabel('Q')
plt.tight_layout()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(5,2.5), dpi=300)

ax = axes[0]
sns.histplot(loopt.loc[sig_loops1].mean(axis=1), ax=ax,color='chocolate',alpha=0.6)
sns.histplot(loopt.loc[sig_loops2].mean(axis=1), ax=ax,color='cadetblue',alpha=0.6)
ax.set_xlabel('T')
ax = axes[1]
sns.histplot(loopt.loc[selpos].mean(axis=1), ax=ax,color='chocolate',alpha=0.6)
sns.histplot(loopt.loc[selneg].mean(axis=1), ax=ax,color='cadetblue',alpha=0.6)
ax.set_xlabel('T')
plt.tight_layout()

In [ ]:
loopq.head(),loopq.shape

## Plot Heatmap

In [ ]:
usecols=list(set(loopq.columns.tolist()) & set(loopt.columns.tolist()) & set(loop_avg_cg.columns.tolist()) & set(group_order))
if not pseudobulk_atac_adata_path is None:
    usecols=list(set(usecols) & set(loop_avg_atac.columns.tolist()))
    loop_avg_atac=loop_avg_atac.loc[:,usecols]

In [ ]:
loopq=loopq.loc[:,usecols]
loopt=loopt.loc[:,usecols]
loop_avg_cg=loop_avg_cg.loc[:,usecols]
loop_avg_ch=loop_avg_ch.loc[:,usecols]

In [ ]:
group_name=group_col if stratify_col is None else 'GroupID'
df_cols=loopq.columns.to_frame(name=group_name)
if not stratify_col is None:
    df_cols[stratify_col]=df_cols.GroupID.apply(lambda x:x.split('+')[0])
    df_cols[group_col]=df_cols.GroupID.apply(lambda x:x.split('+')[1])
if not parent_col is None:
    df_cols[parent_col]=df_cols[group_name].map(group2parent)
    df_cols.sort_values([parent_col,group_name],inplace=True)
else:
    if split_order is None:
        df_cols.sort_values(group_name,inplace=True)
    else:
        df_cols=df_cols.loc[split_order]
df_cols

In [ ]:
cols=loopq.columns.tolist()
max_idx=np.argmax(loopq.values,axis=1)
df_loop['Group']=[cols[i] for i in max_idx]
df_loop

In [ ]:
print(thres1,thres2)
sig_loops = ((zscore(df_loop[f'{use_group}.Q.F'])>thres2) & (zscore(df_loop[f'{use_group}.T.F'])>thres2))
print(sig_loops.sum())

In [ ]:
df_rows=df_loop.loc[sig_loops]
if not stratify_col is None:
    df_rows[stratify_col]=df_rows.Group.apply(lambda x:x.split('+')[0])
    df_rows[group_col]=df_rows.Group.apply(lambda x:x.split('+')[1])
if not parent_col is None:
    df_rows[parent_col]=df_rows.Group.map(group2parent)
df_rows.head()

In [ ]:
row_order=[]
if stratify_col is None:
    for g in df_cols[group_col].tolist():
        rows=df_rows.loc[df_rows.Group==g].sort_values(f'{group_col}.Q.F',ascending=False).index.tolist()
        row_order.extend(rows)
else:
    parent2stratify=df_rows.loc[:,[parent_col,stratify_col]].drop_duplicates().groupby(parent_col)[stratify_col].apply(lambda x:x.unique().tolist()).to_dict()
    for g2 in df_cols['GroupID'].unique():
        rows=df_rows.loc[(df_rows['Group']==g2)].sort_values(f'{stratify_col}.Q.F',ascending=False).index.tolist()
        row_order.extend(rows)
df_rows=df_rows.loc[row_order]

In [ ]:
from pym3c.utils import read_markers
df_marker=read_markers(
        marker_path,marker_sheet,
        species=species,marker_cols = ['Methylation_markers','SelectedMarkers'] #'RNA_markers'
    )
marker_genes=list(set(df_marker.Markers.sum()))
df_marker.head()

In [ ]:
if not gene_annotation is None:
    df1=df_rows.iloc[:,:6].rename(columns={'chr1':'chr','start1':'start','end2':'end'})
    df1.drop(['chr2','end1','start2'],inplace=True,axis=1)
    df1['ID']=df1['chr'].map(str)+'-'+df1.start.map(str)+'-'+df1.end.map(str)
    bed1 = pybedtools.BedTool.from_dataframe(df1)
    df2=pd.read_csv(os.path.expanduser(gene_annotation),sep='\t',header=None,names=['chr','start','end','gene_id','gene_symbol','strand','gene_type'])
    bed2 = pybedtools.BedTool.from_dataframe(df2)
    intersection = bed1.intersect(bed2, wa=True, wb=True,loj=True) #f=0.7,F=0.7,r=True
    df=intersection.to_dataframe(names=df1.columns.tolist()+['chr2','start2','end2']+df2.columns.tolist()[3:])
    # read dmg table
    # df_dmg=pd.read_csv(os.path.expanduser(dmg),sep='\t')
    # df_dmg=df_dmg.loc[(df_dmg.wilcox_statistic<0) & (df_dmg.Group==group_col)]
    # gene_list=df_dmg.Gene.unique().tolist()
    df=df.loc[df.gene_type=='protein_coding']
    if not marker_path is None:
        df=df.loc[df.gene_symbol.isin(marker_genes)]
    df['length_intersection'] = df[['end', 'end2']].min(axis=1) - df[['start', 'start2']].max(axis=1)
    df['fraction1']=df['length_intersection'] / (df.end-df.start)
    df['fraction2']=df['length_intersection'] / (df.end2-df.start2)
    df['fraction']=df.loc[:,['fraction1','fraction2']].min(axis=1)
    df.sort_values('fraction',ascending=False,inplace=True)
    df=df.loc[df.fraction >= 0.95 ]
    df.drop_duplicates('ID',keep='first',inplace=True)
    
    D=df.groupby('ID').gene_symbol.apply(lambda x:','.join(x.unique().tolist()) if x.nunique() > 0 else '').to_dict()
    df_rows['ID']=df_rows['chr1'].map(str)+'-'+df_rows['start1'].map(str)+'-'+df_rows['end2'].map(str)
    df_rows['Gene']=df_rows.ID.map(D)
    print(df_rows.Gene.nunique())
df_rows

In [ ]:
tmpcg = loop_avg_cg.loc[df_rows.index.tolist(),df_cols.index.tolist()]
tmpch = loop_avg_ch.loc[df_rows.index.tolist(),df_cols.index.tolist()]
tmp3c = loopq.loc[df_rows.index.tolist(),df_cols.index.tolist()]
if not pseudobulk_atac_adata_path is None:
    atac=loop_avg_atac.loc[df_rows.index,df_cols.index.tolist()]

In [ ]:
if not parent_col is None:
    tmp_colors={}
    for p in df_rows[parent_col].unique():
        tmp_colors[p]=parent_colors[p]
    row_ha_dict={parent_col:anno_simple(df_rows[parent_col],colors=tmp_colors,legend=False)}
    row_ha = HeatmapAnnotation(**row_ha_dict,axis=0,
             verbose=0,label_kws=dict(visible=False))
    row_split=df_rows[parent_col]
    row_split_order=df_rows[parent_col].unique().tolist() if split_order is None else [o for o in split_order if o in df_rows[parent_col].tolist()]
else:
    row_ha=None
    row_split=None
    row_split_order=None

if not gene_annotation is None:
    right_ha=HeatmapAnnotation(
        Gene=anno_label(df_rows.Gene,merge=True,extend=True,colors='black',fontsize=6,relpos=(0,0.5)),
        axis=0,verbose=0,orientation='right')
else:
    right_ha=None
show_colnames=True
if not parent_col is None:
    tmp_colors={}
    for p in df_cols[parent_col].unique():
        tmp_colors[p]=parent_colors[p]
    col_ha_dict={
                parent_col:anno_simple(
                    df_cols[parent_col],#height=5,
                     colors=tmp_colors,
                     add_text=False,text_kws={'fontweight':'bold','color':'white'},legend=True),
                }
    if stratify_col in df_cols.columns.tolist():
        tmp_colors={}
        for p in df_cols[stratify_col].unique():
            tmp_colors[p]=stratify_colors[p]
        col_ha_dict[stratify_col]=anno_simple(df_cols[stratify_col],colors=tmp_colors,legend=True)
        tmp_colors={}
        for p in df_cols[group_col].unique():
            tmp_colors[p]=group_colors[p]
        col_ha_dict[group_col]=anno_simple(df_cols[group_col],colors=tmp_colors,legend=True)
        show_colnames=False

    col_ha = HeatmapAnnotation(**col_ha_dict,axis=1,
             verbose=0,label_kws=dict(visible=False))
    col_split=df_cols[parent_col]
    col_split_order=df_cols[parent_col].unique().tolist() if split_order is None else [o for o in split_order if o in df_cols[parent_col].tolist()]
else:
    col_ha=None
    col_split=None
    col_split_order=None

cm1 = ClusterMapPlotter(
            data=tmp3c, top_annotation=col_ha, left_annotation=row_ha,
            show_rownames=False, show_colnames=show_colnames,col_cluster=False,row_cluster=False,
            row_dendrogram=False, col_dendrogram=False,
            row_cluster_method='ward',row_cluster_metric='cosine',
            col_split=col_split, col_split_order=col_split_order,
            xticklabels_kws={'labelrotation':-60,'labelcolor':'black','labelsize':10},xticklabels=True,
            col_split_gap=0,cmap='BlueDarkRed18',#turbo, parula, viridis,RedYellowBlue_r,BlueOrangeRed,
            # row_split=df_rows.Group,row_split_order=df_cols.MajorType.tolist(),
            row_split=row_split,row_split_gap=0,row_split_order=row_split_order,
            rasterized=True, #row_split_gap=0.1,vmax=1,vmin=0,center=0.5,
            plot=False,label='Interaction',
            standard_scale=0,legend_kws=dict(extend='both',extendfrac=0.1),
            verbose=1)


cm2 = ClusterMapPlotter(
            data=tmpcg, top_annotation=col_ha, #left_annotation=row_ha,
            show_rownames=False, show_colnames=show_colnames,col_cluster=False,row_cluster=False,
            row_dendrogram=False, col_dendrogram=False,xticklabels=True,
            col_split=col_split, col_split_order=col_split_order,
            xticklabels_kws={'labelrotation':-60,'labelcolor':'black','labelsize':10},
            col_split_gap=0,row_split_gap=0,rasterized=True,
            plot=False,cmap='parula',label='mCG',
            standard_scale=0,legend_kws=dict(extend='both',extendfrac=0.1),
            verbose=0)

cm3 = ClusterMapPlotter(
            data=tmpch, top_annotation=col_ha, right_annotation=right_ha,
            show_rownames=False, show_colnames=show_colnames,col_cluster=False,row_cluster=False,
            row_dendrogram=False, col_dendrogram=False,xticklabels=True,
            col_split=col_split, col_split_order=col_split_order,
            xticklabels_kws={'labelrotation':-60,'labelcolor':'black','labelsize':10},
            col_split_gap=0,row_split_gap=0,rasterized=True,
            plot=False,cmap='BlueOrangeRed',label='mCH',
            standard_scale=0,legend_kws=dict(extend='both',extendfrac=0.1),
            verbose=0) #or Greys,BlueOrange12,

if not pseudobulk_atac_adata_path is None:
    cm4 = ClusterMapPlotter(
            data=atac, top_annotation=col_ha, 
            show_rownames=False, show_colnames=show_colnames,col_cluster=False,row_cluster=False,
            row_dendrogram=False, col_dendrogram=False,xticklabels=True,
            col_split=col_split, col_split_order=col_split_order,
            xticklabels_kws={'labelrotation':-60,'labelcolor':'black','labelsize':10},
            col_split_gap=0,row_split_gap=0,rasterized=True,
            plot=False,cmap='BlueDarkOrange12',label='ATAC\nlog(CPM)',
            standard_scale=0,legend_kws=dict(extend='both',extendfrac=0.1),
            verbose=0)
    cmlist=[cm1,cm4,cm2,cm3]
else:
    cmlist=[cm1,cm2,cm3]
# cm3.top_annotation.label_kws['visible']=True

In [ ]:
from PyComplexHeatmap.utils import despine
nrows,ncols=tmp3c.shape
width=max(ncols*0.5,5) if pseudobulk_atac_adata_path is None else max(ncols*0.8,5)
plt.figure(figsize=(width,10))
ax,legend_axes=composite(cmlist=cmlist, main=0,legend_hpad=30,col_gap=3.5,legend_gap=10)
cm1.ax.set_title("Loop Q",fontsize=14)
cm2.ax.set_title("Avg mCG",fontsize=14)
cm3.ax.set_title("Avg mCH",fontsize=14)
if not pseudobulk_atac_adata_path is None:
    cm4.ax.set_title("Avg ATAC",fontsize=14)
ax.set_ylabel(str(tmp3c.shape[0])+" Diff Loops",color='black',fontsize=14)
ax.set_axis_on()
despine(ax=ax, left=True, bottom=True,top=True,right=True)
ax.tick_params(axis="both",which="both",left=False,
                right=False,labelleft=False,labelright=False,
                top=False,bottom=False,labeltop=False,labelbottom=False)
# ax.set_title("Title",y=1.03,fontdict={'fontweight':'bold','color':'red'})
plt.savefig(os.path.join(figdir,f'{group_col}.diff_loop.multiple_omics_heatmap.pdf'), transparent=True,bbox_inches='tight')
plt.show()

In [ ]:
# # calculate pearson correlation for each cell type across all bins
# use_cols=[ct for ct in group_order if ct in loop_avg_cg.columns.tolist()]
# mcg=loop_avg_cg.loc[:,use_cols]
# mch=loop_avg_ch.loc[:,use_cols]
# atac=loop_avg_atac.loc[:,use_cols]
# # rows=df_rows.index.tolist()
# rows=df_loop.index.tolist() # using all loops to calculate correlation
# print(len(rows))
# R=[]
# for ct in cols:
#     r_cg=pearsonr(loopq.loc[rows,ct].values, mcg.loc[rows,ct].values)
#     r_ch=pearsonr(loopq.loc[rows,ct].values, mch.loc[rows,ct].values)
#     r_atac=pearsonr(loopq.loc[rows,ct].values, atac.loc[rows,ct].values)
#     R.append([ct,r_cg[0],r_cg[1],r_ch[0],r_ch[1],r_atac[0],r_atac[1]])
# data=pd.DataFrame(R,columns=['CellType','CG_corr','CG_pvalue','CH_corr','CH_pvalue','ATAC_corr','ATAC_pvalue']).set_index('CellType')
# data.to_csv(os.path.join(figdir,f'{group_col}.multiomics_correlation_across_bins.tsv'),sep='\t')

# df1=data.loc[:,['CG_corr','CH_corr','ATAC_corr']]
# df1.columns=[col.replace('_corr','') for col in df1.columns.tolist()]

# row_info=pd.Series(group2parent)
# row_info=row_info.loc[df1.index.tolist()]
# colors={}
# for parent in row_info.values:
#     if parent not in row_info.tolist():
#         continue
#     colors[parent]=parent_colors[parent]
# row_ha_dict={
#     parent_col:anno_simple(row_info,colors=colors,legend=False),
# }
# row_ha = HeatmapAnnotation(**row_ha_dict,axis=0,
#          verbose=0,label_side='bottom',label_kws=dict(visible=False))

# plt.figure(figsize=(3,5))
# cm = ClusterMapPlotter(
#                 data=df1, left_annotation=row_ha,
#                 show_rownames=True, show_colnames=True,col_cluster=False,row_cluster=True,
#                 row_dendrogram=False, col_dendrogram=False,
#                 row_split=row_info,xticklabels=True,yticklabels=True,
#                 xticklabels_kws={'labelrotation':-30,'labelcolor':'black','labelsize':10,'bottom':True},
#                 yticklabels_kws=dict(right=True),
#                 row_split_gap=0.5,rasterized=True,#linecolor='white',linewidth=0.05,
#                 cmap='Spectral_r',label='Correlation',annot=True,
#                 legend_kws=dict(extend='both',extendfrac=0.1),# standard_scale=1,
#                 verbose=0)
# plt.savefig(os.path.join(figdir,f'{group_col}.multiomics_correlation_across_bins.pdf'), 
#             transparent=True,bbox_inches='tight',dpi=300)
# plt.show()

In [ ]:
order=df_rows.Group.value_counts().index.tolist()
plt.figure(figsize=(8,3.5))
ax=sns.countplot(df_rows,x='Group',order=order,palette='Spectral')
ax.set_title("No. of diff loops",fontsize=14)
ax.xaxis.label.set_visible(False)
plt.setp(plt.gca().get_xticklabels(), rotation=-45, ha='left')
plt.savefig(os.path.join(figdir,f'No.of_diff_loop_per_{group_col}_countplot.pdf'), transparent=True,bbox_inches='tight')
plt.show()
df_rows.sort_values(f'{use_group}.Q.F',ascending=False,inplace=True)
# df_rows.loc[(df_rows.Gene.notna()) & ((df_rows.mCG_corr.abs()>=0.7) | (df_rows.mCH_corr.abs()>=0.7))]

In [ ]:
fig, ax = plt.subplots(figsize=(2,2), dpi=300)
if 'ATAC_corr' in df_rows.columns.tolist():
    plot_data=[
        df_rows['mCG_corr'],
        df_rows['mCH_corr'],
        df_rows['ATAC_corr'],
              ]
    xticklabels=['mCG', 'mCH', 'ATAC']
    colors=['limegreen','royalblue','darkorange']
else:
    plot_data=[
        df_rows['mCG_corr'],
        df_rows['mCH_corr']
    ]
    xticklabels=['mCG', 'mCH']
    colors=['limegreen','royalblue']
ax.set_yticks([-1,-0.5, 0,0.5,1])
ax.grid(axis='y',which='major',zorder=-1,linestyle='--')
sns.violinplot(plot_data, ax=ax,palette=colors)
ax.set_xticklabels(xticklabels)
# ax.set_title(group,fontsize=10)
ax.set_ylabel('Correlation with Loop A',fontsize=8) #PCC with Interaction
plt.savefig(os.path.join(figdir,f'{group_col}_multiomics_correlation_violinplot.pdf'), transparent=True,bbox_inches='tight')
plt.show()

In [ ]:
df_rows.to_csv(os.path.join(figdir,"diff_loops.tsv"),sep='\t')